In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =====================================================
# 1. Load Dataset
# =====================================================

df = pd.read_csv("Deposit_data.csv")

# Keep a copy before encoding
df_original = df.copy()

# =====================================================
# 2. Data Cleaning
# =====================================================

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove ID column
if "Id" in df.columns:
    df.drop(columns=["Id"], inplace=True)

# Replace invalid age
df["age"] = df["age"].replace(999, np.nan)

# Fill missing numeric values
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Replace "unknown" with NaN
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].replace("unknown", np.nan)
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nMissing Values:")
print(df.isnull().sum())

# =====================================================
# 3. Encode Categorical Columns
# =====================================================

encoders = {}

categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))
    encoders[col] = encoder

# =====================================================
# 4. Split Features and Target
# =====================================================

X = df.drop("y", axis=1)
y = df["y"]

# =====================================================
# 5. Feature Scaling
# =====================================================

scaler = StandardScaler()
X = scaler.fit_transform(X)

# =====================================================
# 6. Train/Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# 7. Create SVM Model
# =====================================================

model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    random_state=42
)

# =====================================================
# 8. Train Model
# =====================================================

model.fit(X_train, y_train)

# =====================================================
# 9. Predict Test Data
# =====================================================

y_pred = model.predict(X_test)

# =====================================================
# 10. Evaluation
# =====================================================

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# =====================================================
# 11. Predict New Customer
# =====================================================

sample = pd.DataFrame([{
    "age": 35,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 1200,
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "may",
    "duration": 250,
    "campaign": 2,
    "pdays": -1,
    "previous": 0,
    "poutcome": "unknown"
}])

# =====================================================
# 12. Clean Sample
# =====================================================
for col in sample.select_dtypes(include="object").columns:

    # Replace unknown values
    if sample[col].iloc[0] == "unknown":
        sample[col] = df_original[col].mode()[0]

    # Check if value exists in encoder
    if sample[col].iloc[0] not in encoders[col].classes_:
        sample[col] = encoders[col].classes_[0]

    # Encode
    sample[col] = encoders[col].transform(sample[col].astype(str))

# =====================================================
# 13. Scale Sample
# =====================================================

sample = scaler.transform(sample)

# =====================================================
# 14. Predict
# =====================================================

prediction = model.predict(sample)

print("\nPrediction:")

if prediction[0] == 1:
    print("Customer WILL subscribe to the deposit.")
else:
    print("Customer will NOT subscribe to the deposit.")